In [1]:
import pandas as pd

train_df = pd.read_csv(r"D:\hackathons & other projects\ai-chat-harassment-detection\data\processed\train_processed.csv")
val_df   = pd.read_csv(r"D:\hackathons & other projects\ai-chat-harassment-detection\data\processed\val_processed.csv")

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

train_df.head()

Train shape: (143613, 2)
Val shape: (15958, 2)


,comment_text,harassment
0,", 25 March 2013 (UTC)\n\nThat's some strange i...",0
1,"""\n{| style=""""background-color: #F5FFFA; paddi...",0
2,You Republic of Turkey and supporters therof a...,1
3,I have commented on them in my unblock reason....,0
4,Very interesting comments about the purpose an...,0


In [2]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['comment_text', 'harassment'],
    num_rows: 143613
})
Dataset({
    features: ['comment_text', 'harassment'],
    num_rows: 15958
})


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

print("RoBERTa tokenizer loaded")

RoBERTa tokenizer loaded


In [ ]:
train_dataset = train_dataset.rename_column("harassment", "labels")
val_dataset   = val_dataset.rename_column("harassment", "labels")

print(train_dataset.column_names)

In [4]:
def tokenize_function(example):

    return tokenizer(
        example["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)

val_dataset = val_dataset.map(tokenize_function, batched=True)


print("Tokenization complete")

Map:   0%|          | 0/143613 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

Tokenization complete


In [6]:
# Rename harassment → labels

train_dataset = train_dataset.rename_column("harassment", "labels")
val_dataset   = val_dataset.rename_column("harassment", "labels")

print("Column renamed to labels")

Column renamed to labels


In [7]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("PyTorch format set")

PyTorch format set


In [7]:
import os
print(os.getcwd())

D:\hackathons & other projects\ai-chat-harassment-detection\notebooks


In [8]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

model.to(device)

print("RoBERTa model loaded on", device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded on cuda


In [9]:
from transformers import TrainingArguments
import os

# Required for new transformers versions
os.environ["TENSORBOARD_LOGGING_DIR"] = "../logs-roberta"

training_args = TrainingArguments(

    # Where model will be saved
    output_dir="../models/roberta-harassment",

    # Learning rate (standard for RoBERTa fine-tuning)
    learning_rate=2e-5,

    # SAFE batch size for 8GB GPU
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Number of epochs
    num_train_epochs=1,

    # Enable mixed precision (VERY important for RTX 4060)
    fp16=True,

    # New syntax (IMPORTANT — old one deprecated)
    eval_strategy="epoch",
    save_strategy="epoch",

    # Load best model automatically
    load_best_model_at_end=True,

    # Disable wandb
    report_to="none"
)

print("TrainingArguments ready")

TrainingArguments ready


In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {

        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall

    }

print("Metrics ready")

Metrics ready


In [11]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics

)

print("Trainer ready")

Trainer ready


In [12]:
# Step 3 — Start RoBERTa Training

trainer.train()

print("\nTraining completed")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.141319,0.153544,0.964532,0.813201,0.875622,0.759088


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training completed


In [3]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "../models/roberta-harassment/checkpoint-35904"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
model.save_pretrained("../models/roberta-harassment-v2")

tokenizer.save_pretrained("../models/roberta-harassment-v2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/roberta-harassment-v2\\tokenizer_config.json',
 '../models/roberta-harassment-v2\\tokenizer.json')

In [8]:
# STEP 1 — Load libraries
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight


# STEP 2 — Load processed CSV (IMPORTANT: use ../ because you are inside notebooks folder)

train_df = pd.read_csv("../data/processed/train_processed.csv")
val_df   = pd.read_csv("../data/processed/val_processed.csv")

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)


# STEP 3 — Convert to HuggingFace Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

print("Dataset conversion successful")


# STEP 4 — Compute class weights

labels = train_df["harassment"]

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = np.array(class_weights, dtype=np.float32)

print("\nClass Weights:")
print(class_weights)

Train shape: (143613, 2)
Val shape: (15958, 2)
Dataset conversion successful

Class Weights:
[0.55659205 4.9175797 ]
